# Feline Skin Disease Detection - Cross-Validated CNN Sweep

Trains every architecture under both transfer-learning strategies across all 5
group-aware folds and 3 seeds (7 x 2 x 5 x 3 = **210 runs**).

Folds come from `fold_assignments.csv` (built by `make_folds.py`), so
near-duplicate images never straddle a train/test boundary.

Every run saves its test and validation probabilities, which is what the
statistical comparison scores. Weights are kept for **one canonical run per
architecture x strategy** (14 models) so Grad-CAM has something to work with;
the other 196 are discarded.

Run all cells in order. Make sure you have a **GPU runtime** enabled:
**Runtime -> Change runtime type -> GPU**

## 1. Dependencies & GPU

`base_classifier.py` imports `ml_insights` at module scope (for the reliability
diagrams in `calibrate_and_evaluate`), so the import fails without it even
though this notebook never calls that method. Colab has everything else
preinstalled.

In [ ]:
!pip install -q ml_insights

import tensorflow as tf

print("TensorFlow version:", tf.__version__)
gpus = tf.config.list_physical_devices('GPU')
print(f"GPUs available: {len(gpus)}")
for gpu in gpus:
    print(" ", gpu)

if not gpus:
    print("\n*** NO GPU. 210 runs on CPU will not finish. "
          "Runtime -> Change runtime type -> GPU ***")

## 2. Mount Google Drive & Clone Repo

In [ ]:
import os
import subprocess
import sys
import time

from google.colab import drive
drive.mount('/content/drive')

# Ensure we're in a valid directory before cleanup
os.chdir('/content')
!rm -rf /content/repo

REPO_URL = "https://github.com/pelta-ai/feline-skin-disease-detection.git"
# Branch holding the current base_classifier / constants - must be pushed to the remote
BRANCH = "feature/duplicate-image-audit"

# GIT_LFS_SKIP_SMUDGE keeps the clone from failing on LFS-tracked .keras files
clone = subprocess.run(
    ["git", "clone", "-b", BRANCH, REPO_URL, "/content/repo"],
    env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
    capture_output=True, text=True,
)
print(clone.stderr.strip())
if clone.returncode != 0:
    raise RuntimeError(f"Clone of branch '{BRANCH}' failed - has it been pushed to the remote?")
assert os.path.isdir('/content/repo/src'), "Checkout incomplete: /content/repo/src is missing."

# Every path below is absolute or relative to the repo root, so it does not matter
# where this notebook itself lives (src/colab_notebooks/) - only the repo cwd.
os.chdir('/content/repo')
sys.path.insert(0, '/content/repo')

# Take the dataset folder name from the repo's own constants (now "new_data")
# so the symlink always matches the relpaths in fold_assignments.csv.
from src.utils import constants
DATA_DIR = constants.DATA_PATH
print(f"Classifiers expect the dataset at: {DATA_DIR}")

DRIVE_ROOT = "/content/drive/MyDrive/feline-skin-disease-detection"
# Drive may still hold the dataset under its old name, so accept either.
DRIVE_DATA = next(
    (os.path.join(DRIVE_ROOT, name)
     for name in (DATA_DIR, "final_data")
     if os.path.isdir(os.path.join(DRIVE_ROOT, name))),
    None,
)
if DRIVE_DATA is None:
    raise FileNotFoundError(
        f"No dataset folder in {DRIVE_ROOT} (looked for '{DATA_DIR}' and 'final_data')."
    )

## 3. Get Dataset onto Local Disk

Colab's Drive mount is a FUSE filesystem: fine for a handful of large files,
very slow for thousands of small ones. Reading images straight off it would
stream the whole dataset over that mount **once per epoch**, and this sweep runs
thousands of epochs. So the dataset lives on the VM's local disk, and the repo's
`new_data` symlink points there.

Three paths, in order of preference:

| Situation | What happens | Cost |
|---|---|---|
| Local copy already present | nothing | - |
| `new_data.tar` on Drive | extract it | one sequential read |
| Neither | `copytree` from Drive, then build the tar for next time | slow once |

Local disk is wiped on every runtime reset, so this cell re-runs each session -
but from the second session on it takes the tar path. Outputs still go to Drive,
so the resume log survives.

**If the dataset on Drive changes, delete `new_data.tar`** or you will keep
extracting the stale copy.

In [ ]:
import os
import shutil
import subprocess
import time

LOCAL_DATA = "/content/dataset"

# Uncompressed .tar on purpose: JPEGs are already compressed, so gzip would burn
# CPU in both directions for a negligible size win. The speedup comes purely from
# one large sequential Drive read replacing 7.6k tiny ones.
ARCHIVE = os.path.join(DRIVE_ROOT, f"{os.path.basename(DRIVE_DATA)}.tar")

# After a cold copy, write the archive back to Drive so later sessions take the
# fast path. Set False to leave Drive untouched.
MAKE_ARCHIVE = True


def _count_images(root):
    return sum(
        1
        for split in ("train", "val", "test")
        for _, _, fns in os.walk(os.path.join(root, split))
        for fn in fns
        if fn.lower().endswith((".jpg", ".jpeg", ".png"))
    )


def _sh(cmd):
    """Run a shell command, raise on failure, return elapsed seconds."""
    t0 = time.time()
    p = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if p.returncode != 0:
        raise RuntimeError(f"{cmd}\n{p.stderr.strip()}")
    return time.time() - t0


n_local = _count_images(LOCAL_DATA) if os.path.isdir(LOCAL_DATA) else 0

if n_local:
    print(f"{LOCAL_DATA} already populated ({n_local} images), nothing to do")

elif os.path.exists(ARCHIVE):
    print(f"Extracting {ARCHIVE} ({os.path.getsize(ARCHIVE) / 1e9:.2f} GB)...")
    os.makedirs(LOCAL_DATA, exist_ok=True)
    secs = _sh(f'tar -xf "{ARCHIVE}" -C "{LOCAL_DATA}"')
    print(f"  extracted in {secs:.0f}s")

else:
    print(f"No archive at {ARCHIVE}")
    print(f"Cold copy {DRIVE_DATA} -> {LOCAL_DATA} (slow, one time)...")
    t0 = time.time()
    shutil.copytree(DRIVE_DATA, LOCAL_DATA, dirs_exist_ok=True)
    print(f"  copied in {time.time() - t0:.0f}s")

    if MAKE_ARCHIVE:
        # Build from the LOCAL copy - re-reading Drive would cost as much as the
        # copy just did. Write to .part and rename, so a disconnect mid-write
        # cannot leave a truncated archive that a later session would happily
        # extract as if it were the whole dataset.
        print(f"Building {ARCHIVE} for future sessions...")
        secs = _sh(f'tar -cf "{ARCHIVE}.part" -C "{LOCAL_DATA}" .')
        os.replace(f"{ARCHIVE}.part", ARCHIVE)
        print(f"  wrote {os.path.getsize(ARCHIVE) / 1e9:.2f} GB in {secs:.0f}s")

n_local = _count_images(LOCAL_DATA)
print(f"\n{n_local} images on local disk")
!du -sh {LOCAL_DATA}
!df -h /content | tail -1

# Point the repo's dataset path at the LOCAL copy, not Drive. Everything
# downstream resolves relpaths against this, so it never has to know the
# difference.
!rm -rf /content/repo/{DATA_DIR}
!ln -s {LOCAL_DATA} /content/repo/{DATA_DIR}
print(f"\nLinked {LOCAL_DATA} -> /content/repo/{DATA_DIR}")

## 4. Load Fold Assignments

`fold_assignments.csv` carries one row per image with `fold` (0-4) and `role`
(train/val/test). Its `relpath` column is `split/class/filename` relative to the
dataset root, which now resolves to the local copy via the symlink above.

The existence check below is the verification that the copy landed intact - it
stats all 7.6k paths, which is fast now that they are local.

In [ ]:
import os
import shutil

import pandas as pd

DRIVE_OUT = "/content/drive/MyDrive/feline-skin-disease-detection/cv_run"
DRIVE_MODELS = os.path.join(DRIVE_OUT, "models")
os.makedirs(DRIVE_OUT, exist_ok=True)
os.makedirs(DRIVE_MODELS, exist_ok=True)

# make_folds.py writes the CSV next to itself; fall back to Drive if this clone
# predates it being committed.
FOLD_CANDIDATES = [
    os.path.join("src", "duplicate_image_audit", "fold_assignments.csv"),
    "fold_assignments.csv",
    os.path.join(DRIVE_ROOT, "fold_assignments.csv"),
]
FOLD_CSV = next((p for p in FOLD_CANDIDATES if os.path.exists(p)), None)
if FOLD_CSV is None:
    raise FileNotFoundError(
        "fold_assignments.csv not found. Either commit it to the repo or drop a "
        f"copy at {DRIVE_ROOT}/. Looked in: {FOLD_CANDIDATES}"
    )
print(f"Using folds from: {FOLD_CSV}")

# Park a copy alongside the results. The probs are meaningless without knowing
# exactly which rows were in which fold, and this clone's copy disappears with
# the runtime.
shutil.copy2(FOLD_CSV, os.path.join(DRIVE_OUT, "fold_assignments.csv"))
print(f"Copied folds -> {os.path.join(DRIVE_OUT, 'fold_assignments.csv')}")

folds = pd.read_csv(FOLD_CSV)
folds["path"] = folds["relpath"].apply(lambda r: os.path.join(DATA_DIR, *r.split("/")))

# Stat every file once up front - a missing image only surfaces mid-epoch otherwise.
missing = folds.loc[~folds["path"].map(os.path.exists), "path"]
if len(missing):
    raise FileNotFoundError(
        f"{len(missing)} of {len(folds)} images missing, e.g.:\n  "
        + "\n  ".join(missing.head(5))
    )

print(f"\n{len(folds)} rows, {folds['label'].nunique()} classes, "
      f"{folds['fold'].nunique()} folds, {folds['group_id'].nunique()} groups")
print(folds.groupby(["fold", "role"]).size().unstack(fill_value=0))

## 5. Sweep Configuration

`name=arch` is what `train_one_run` stamps into every output filename. Without
it `self.name` is `None` and all seven architectures overwrite each other's
files. Artifacts land as:

```
preds_mobilenet_v2_frozen_f0_s1.npy       test-set probabilities
preds_val_mobilenet_v2_frozen_f0_s1.npy   val-set probabilities (for calibration)
testset_f0.csv                            the fold's test rows (arch-independent)
models/mobilenet_v2_frozen_f0_s1.keras    weights, kept for the one canonical run
```

Weights are kept only for `KEEP_FOLD`/`KEEP_SEED` - 14 models (one per
architecture x strategy) instead of 210, which is roughly 0.6 GB of Drive rather
than 9-12 GB. That is enough to run Grad-CAM on every arch/strategy combination
while still scoring all 210 runs from their saved probabilities.

In [ ]:
ARCHS = ["mobilenet_v2", "mobilenet_v3_small", "resnet50", "efficientnet_b0",
         "efficientnet_v2_b0", "nasnet_mobile", "convnext_tiny"]

CONFIGS = [
    ("frozen",    dict(trainable_backbone=False, learning_rate=1e-3)),
    ("finetuned", dict(trainable_backbone=True, unfreeze_last_n_layers=30,
                       learning_rate=1e-5)),
]

SEEDS = (1, 2, 3)
N_FOLDS = folds["fold"].nunique()

# Keep the weights from exactly one (fold, seed) per arch x strategy, for
# Grad-CAM. Every other run is scored from its probabilities and discarded.
KEEP_FOLD, KEEP_SEED = 0, 1

n_runs = len(ARCHS) * len(CONFIGS) * N_FOLDS * len(SEEDS)
n_kept = len(ARCHS) * len(CONFIGS)
print(f"{len(ARCHS)} archs x {len(CONFIGS)} strategies x {N_FOLDS} folds "
      f"x {len(SEEDS)} seeds = {n_runs} runs")
print(f"keeping weights for fold {KEEP_FOLD} / seed {KEEP_SEED} only -> "
      f"{n_kept} models (~0.6 GB), the other {n_runs - n_kept} are discarded")

## 6. Parameter Counts

Sanity-check what `finetuned` actually unfreezes before spending GPU hours on it.
`unfreeze_last_n_layers=30` counts Keras *layers* from the end of the backbone,
so the same 30 means very different trainable fractions across architectures
(ConvNeXtTiny's last 30 layers are far heavier than MobileNetV3Small's).

This downloads every set of ImageNet weights, so the first run takes a few minutes.

In [ ]:
import numpy as np
from tensorflow import keras

from src.classifiers import ClassifierFactory

print(f"{'arch':22s} {'strategy':10s} {'total':>11s} {'trainable':>11s} {'%':>6s}")
print("-" * 64)

for arch in ARCHS:
    clf = ClassifierFactory.create(arch, name=arch)
    clf.set_class_names(folds)
    for strategy, kwargs in CONFIGS:
        m = clf._build_model(**kwargs)
        tot = m.count_params()
        tr = sum(int(np.prod(w.shape)) for w in m.trainable_weights)
        print(f"{arch:22s} {strategy:10s} {tot:>11,} {tr:>11,} {100*tr/tot:5.1f}%")
        del m
        keras.backend.clear_session()

## 7. Cross-Validation Sweep

Reads images from local disk; writes probabilities and `cv_results.csv` to Drive
so they survive a runtime reset. `cv_results.csv` doubles as the resume log -
re-running this cell after a reset skips everything already finished.

In [ ]:
import gc
import os
import time

import pandas as pd
from tensorflow import keras

from src.classifiers import ClassifierFactory

# Resume: cv_results.csv is the record of what already ran.
RESULTS_CSV = os.path.join(DRIVE_OUT, "cv_results.csv")
if os.path.exists(RESULTS_CSV):
    results = pd.read_csv(RESULTS_CSV).to_dict("records")
    done = {(r["arch"], r["strategy"], r["fold"], r["seed"]) for r in results}
    print(f"Resuming - {len(done)} of {len(ARCHS) * len(CONFIGS) * N_FOLDS * len(SEEDS)} "
          "runs already complete")
else:
    results, done = [], set()

for arch in ARCHS:
    clf = ClassifierFactory.create(arch, name=arch)
    clf.set_class_names(folds)

    for strategy, kwargs in CONFIGS:
        for k in range(N_FOLDS):
            f = folds[folds.fold == k]
            train_df = f[f.role == "train"]
            val_df   = f[f.role == "val"]
            test_df  = f[f.role == "test"]

            for seed in SEEDS:
                if (arch, strategy, k, seed) in done:
                    continue

                keep = (k == KEEP_FOLD and seed == KEEP_SEED)

                t0 = time.time()
                r = clf.train_one_run(
                    train_df, val_df, test_df, seed,
                    strategy=strategy, fold=k,
                    keep_models=keep,
                    model_save_path=DRIVE_MODELS,
                    probs_save_path=DRIVE_OUT,
                    **kwargs,
                )
                r["seconds"] = round(time.time() - t0, 1)
                r["model_kept"] = keep
                results.append(r)
                pd.DataFrame(results).to_csv(RESULTS_CSV, index=False)
                print(r)

    # Each backbone leaves its graph behind; 210 runs in one process will OOM
    # without this.
    del clf
    keras.backend.clear_session()
    gc.collect()

print(f"\nDone. {len(results)} runs recorded in {RESULTS_CSV}")

## 8. Run Summary

`train_one_run` returns bookkeeping only - accuracy is **not** in `cv_results.csv`.
Metrics get computed from the saved `preds_*.npy` probabilities against
`testset_f*.csv`.

In [ ]:
import pandas as pd

df = pd.read_csv(RESULTS_CSV)
expected = len(ARCHS) * len(CONFIGS) * N_FOLDS * len(SEEDS)
print(f"{len(df)} / {expected} runs complete\n")

print("Runs per architecture x strategy:")
print(df.pivot_table(index="arch", columns="strategy", values="seed",
                     aggfunc="count", fill_value=0), "\n")

print("Median minutes per run, and epochs actually run before early stopping:")
summary = df.groupby(["arch", "strategy"]).agg(
    runs=("seed", "count"),
    median_min=("seconds", lambda s: round(s.median() / 60, 1)),
    total_hours=("seconds", lambda s: round(s.sum() / 3600, 2)),
    median_epochs=("epochs_run", "median"),
)
print(summary)
print(f"\nTotal compute so far: {df['seconds'].sum() / 3600:.1f} hours")

In [ ]:
import os
from collections import Counter

files = os.listdir(DRIVE_OUT)
test_probs = [f for f in files if f.startswith("preds_") and not f.startswith("preds_val_")]
val_probs  = [f for f in files if f.startswith("preds_val_")]
testsets   = [f for f in files if f.startswith("testset_f")]

print(f"{len(test_probs)} test-prob files, {len(val_probs)} val-prob files, "
      f"{len(testsets)} testset CSVs")
print(f"fold_assignments.csv present: "
      f"{os.path.exists(os.path.join(DRIVE_OUT, 'fold_assignments.csv'))}")

# Per-arch counts, parsed back out of the filenames
per_arch = Counter(f[len("preds_"):].rsplit("_", 3)[0] for f in test_probs)
print("\nprob files per arch:")
for a, n in sorted(per_arch.items()):
    print(f"  {a:22s} {n:3d}")

# Grad-CAM keepers: one per arch x strategy
print(f"\nKept models in {DRIVE_MODELS}:")
kept = sorted(f for f in os.listdir(DRIVE_MODELS) if f.endswith(".keras"))
kept_gb = sum(os.path.getsize(os.path.join(DRIVE_MODELS, f)) for f in kept) / 1e9
for f in kept:
    print(f"  {f:52s} {os.path.getsize(os.path.join(DRIVE_MODELS, f)) / 1e6:7.1f} MB")
print(f"  {'':52s} {kept_gb * 1000:7.1f} MB total")

expected_models = {
    f"{a}_{s}_f{KEEP_FOLD}_s{KEEP_SEED}.keras"
    for a in ARCHS for s, _ in CONFIGS
}
absent = sorted(expected_models - set(kept))
if absent:
    print(f"\nMISSING {len(absent)} of {len(expected_models)} Grad-CAM models:")
    for f in absent:
        print(f"  {f}")
    print("(a run finished in an earlier session before keep_models was enabled "
          "will not be re-run by the resume logic - delete its row from "
          "cv_results.csv to force it)")
else:
    print(f"\nAll {len(expected_models)} Grad-CAM models present.")